# OCTO 

In [ ]:
import pymaid

# Load the pymaid configuration

volumes = [
    {
        "name": "MR143",
        "url": "https://neurophyla.mrc-lmb.cam.ac.uk/catmaid/fibsem/",
        "project": "19",
        "token": "x"
    },
    {
        "name": "Octo",
        "url": "https://neurophyla.mrc-lmb.cam.ac.uk/catmaid/fibsem/",
        "project": "18",
        "token": "x"
    }
]
http_user = "x"
http_pass = "x"


In [187]:

rm_octo = pymaid.CatmaidInstance(volumes[1]["url"], project_id=volumes[1]["project"], api_token=volumes[1]["token"],
                                 http_user=http_user, http_password=http_pass)
rm_octo


INFO  : Global CATMAID instance set. Caching is ON. (pymaid)


CatmaidInstance at 6488490384.
Server: https://neurophyla.mrc-lmb.cam.ac.uk/catmaid/fibsem
Project: 18
Caching True (size limit 128; time limit None)
Cache size: 0.0

In [188]:
annotations_list_octo = ["annotation:cube3: pushed fp synapses", "annotation:cube2: pushed fp synapses",
                         "annotation:cube1: pushed fp synapses"]


In [ ]:
import pymaid
import requests
import urllib

import pandas as pd
def get_transactions(range_start=None, range_length=25, remote_instance=None):
    """
    Retrieve transactions from Catmaid. Rewrite pymaid.get_transactions to include millisecond time."""
    remote_instance = pymaid.utils._eval_remote_instance(remote_instance)
    remote_transactions_url = remote_instance._get_transactions_url()

    desc = {'range_start': range_start, 'range_length': range_length}
    desc = {k: v for k, v in desc.items() if v is not None}
    remote_transactions_url += '?%s' % urllib.parse.urlencode(desc)

    data = remote_instance.fetch(remote_transactions_url)
    df = pd.DataFrame.from_dict(data['transactions'])

    user_list = pymaid.get_user_list(remote_instance=remote_instance)
    user_dict = user_list.set_index('id').login.to_dict()
    df['user'] = df.user_id.map(user_dict)

    # Preserve milliseconds and timezone
    df['execution_time'] = pd.to_datetime(df['execution_time'], format='%Y-%m-%dT%H:%M:%S.%f%z')

    return df


def _get_transactions_location_url(self, **GET):
    """Generate url to get transactions (GET)."""
    return self.make_url(self.project_id, 'transactions/location', **GET)

# Add the method to the CatmaidInstance class
setattr(pymaid.CatmaidInstance, '_get_transactions_location_url', _get_transactions_location_url)


def get_transactions_with_locations(remote_instance, project_id, range_length=500):
    """
    Retrieve all transactions (using pymaid.get_transactions) and add location details
    for each transaction.

    Args:
        remote_instance: The remote instance configuration for pymaid.
        project_id (int): The Catmaid project ID.
        range_length (int): How many transactions to retrieve.

    Returns:
        list: List of transaction dictionaries, each enriched with location data.
    """
    # Fetch transactions from pymaid.
    # transactions = pymaid.get_transactions(remote_instance=remote_instance, range_length=range_length)
    transactions = get_transactions(remote_instance=remote_instance, range_length=range_length)
    # print(type(transactions))

    enriched_transactions = []
    for i, transaction in transactions.iterrows(): # transaction is a df
        # print(transaction)
        transaction_id = transaction.transaction_id
        execution_time = transaction.execution_time
        
        # print(transaction_id, execution_time)

        # Make sure the required keys are available.
        if not transaction_id or not execution_time:
            continue

        params = urllib.parse.urlencode({
        "transaction_id": transaction_id,
        "execution_time": execution_time
        })

        # Build URL for location details
        url = f"{remote_instance.server}/{remote_instance.project_id}/transactions/location?{params}"

        # If pymaid has a lower-level GET method, use it:
        try:
            # Use the built-in fetch method to retrieve data.
            location_data = remote_instance.fetch(url, return_type="json")

            # print(location_data)
            #pymaid._get(url, remote_instance=remote_instance)
        except AttributeError:
            # Otherwise, fall back to using requests directly (ensure proper authentication as needed)
            location_response = requests.get(url, headers=pymaid.get_headers(remote_instance))
            location_response.raise_for_status()
            location_data = location_response.json()
        except Exception as e:
            print(f"Error fetching location data for transaction {transaction_id}: {e}")
            location_data = {'x': None,'y': None, 'z': None}

        # Append the location details to the transaction.
        transaction["location_data"] = location_data
        enriched_transactions.append(transaction)
    
    return enriched_transactions


# ----- Usage Example -----

# Set up your remote instance and project id (replace with your actual settings)
rm_octo = pymaid.CatmaidInstance("https://neurophyla.mrc-lmb.cam.ac.uk/catmaid/fibsem/", project_id=18,
                                 api_token="4f640a9240861b412ebcab8cc01f574e5aee16c3",
                                 http_user="smohinta", http_password="headset-recovery-handshake")
project_id = 18

# Now call the function to retrieve enriched transactions.
try:
    txns_with_locations = get_transactions_with_locations(rm_octo, project_id, range_length=500)
    print(txns_with_locations)
    for txn in txns_with_locations:
        print("Transaction ID:", txn.get("transaction_id"))
        print("Location Data:", txn.get("location_data"))
        print("=" * 40)
except Exception as e:
    print("An error occurred:", e)


In [ ]:
import pandas as pd
output_dir = "/Users/sam/Library/CloudStorage/OneDrive-UniversityofCambridge/Synapse_localisation/synapse_curation/catmaid_tracker_plots"

# Convert to DataFrame
df = pd.DataFrame(txns_with_locations)

# Print the DataFrame
display(df)

# Save the DataFrame to a CSV file
df.to_csv(f"{output_dir}/transactions_with_locations.csv", index=False)

# group by user and label and count the number of transactions
grouped = df.groupby(['user', 'label']).size().reset_index(name='counts')
grouped

In [ ]:
# Filter transactions on the hackathon day
import pandas as pd
from datetime import datetime


# Convert execution_time to datetime if it's not already
df['execution_time'] = pd.to_datetime(df['execution_time'])

# Get today's date
today = pd.Timestamp(2025, 3, 6).date() #datetime.now().date()
print(today)

# Filter for today's transactions
today_df = df[df['execution_time'].dt.date == today]

# drop one user- smohinta-su
today_df = today_df[today_df['user'] != 'smohinta-su']
today_df = today_df[today_df['user'] != 'smohinta']


# replace hack_guest with sharris
today_df['user'] = today_df['user'].replace({'hack_guest': 'sharris'})

# replace realnames with anonymised reviewer names
today_df['user_anon'] = today_df['user'].replace({'mclayton': 'Reviewer 1', 'mrobbins': 'Reviewer 2', 'gmo': 'Reviewer 3', 'adulac': 'Reviewer 4',
                                             'nceffa': 'Reviewer 5',
                                             'sharris': 'Reviewer 6',
                                             'swilson': 'Reviewer 7',
                                             'shiyan': 'Reviewer 8',
                                             'hack_guest': 'Reviewer 9'
                                             }) 

display(today_df)


# group by user and label and count the number of transactions
grouped = today_df.groupby(['user', 'label']).size().reset_index(name='counts')
grouped # this looks correct now based on our expectations from talking to the reviewers


In [ ]:
# Filter only annotations:add from transactions because we are interested in synapse annotations

today_df_filt = today_df[today_df['label'] == 'annotations.add']
display(today_df_filt)


# Again group by user and label and count the number of transactions
grouped_filt = today_df_filt.groupby(['user', 'label']).size().reset_index(name='counts')
grouped_filt


In [ ]:
# Save the filtered DataFrame to a CSV file
today_df_filt.to_csv(f"{output_dir}/transactions_with_locations_filtered.csv", index=False)

In [ ]:
# Check how many location_data = {'x': None, 'y': None, 'z': None} are there
display(today_df_filt['location_data'].value_counts())

# Which users have location_data = {'x': None, 'y': None, 'z': None}
location_missing = today_df_filt[today_df_filt['location_data'] == {'x': None, 'y': None, 'z': None}]
# group by user and label and count the number of transactions
grouped_missing = location_missing.groupby(['user', 'label']).size().reset_index(name='counts')
display(grouped_missing)


# Filter out all missing locations
today_df_filt = today_df_filt[today_df_filt['location_data'] != {'x': None, 'y': None, 'z': None}]
display(today_df_filt)

# Extract x, y, z coordinates into separate columns
today_df_filt['post_x'] = today_df_filt['location_data'].apply(lambda x: x['x'])
today_df_filt['post_y'] = today_df_filt['location_data'].apply(lambda x: x['y'])
today_df_filt['post_z'] = today_df_filt['location_data'].apply(lambda x: x['z'])
display(today_df_filt)


# Group by user and label and count the number of transactions
grouped_filt = today_df_filt.groupby(['user', 'label']).size().reset_index(name='counts')
grouped_filt


In [ ]:
# Find all neurons_ids and locations based on annotations list
import pandas as pd
def get_neurons_from_annotations_list(annotations_list, remote_instance):
    """
    Get all neurons and their locations based on a list of annotations.
    """
    neurons = []
    for annotation in annotations_list:
        # Get all annotations with the given name
        annotation_neurons = pymaid.get_neurons(annotation, remote_instance=remote_instance)
        neurons.extend(annotation_neurons)
    return neurons

neurons = get_neurons_from_annotations_list(annotations_list_octo, rm_octo)
print(neurons)

# Assuming these are all post neurons 
post = [neuron.id for neuron in neurons]
for i, neuron in enumerate(neurons):
    node_coords = neuron.nodes[['x', 'y', 'z']]
    print(f"Coordinates for neuron {neuron.skeleton_id}:")
    print(node_coords)
    print("=" * 40)
    if i == 2:
        break


# Create a list to store all neuron data
neuron_data = []

for neuron in neurons:
    # Get coordinates for each node in the neuron
    node_coords = neuron.nodes[['x', 'y', 'z']]
    
    # For each node in the neuron, create a row with all information
    for idx, coords in node_coords.iterrows():
        neuron_data.append({
            'neuron_id': neuron.id,
            'skeleton_id': neuron.skeleton_id,
            'name': neuron.name,
            'x': coords['x'],
            'y': coords['y'],
            'z': coords['z'],
            'node_id': idx
        })

# Convert to DataFrame
post_neurons_all_from_ann_df = pd.DataFrame(neuron_data)
display(post_neurons_all_from_ann_df)

In [ ]:
# Filter the post_neurons_all_from_ann_df based on the post neurons we have from today_df_filt based on locations
# Create a mask for matching coordinates
matching_coords = post_neurons_all_from_ann_df.apply(
    lambda row: any((row['x'] == today_df_filt['post_x']) & 
                   (row['y'] == today_df_filt['post_y']) & 
                   (row['z'] == today_df_filt['post_z'])), 
    axis=1
)

# Filter post_neurons_all_from_ann_df to keep only the matching rows
post_neurons_all_from_ann_df_filterby_today_df = post_neurons_all_from_ann_df[matching_coords]
display(post_neurons_all_from_ann_df_filterby_today_df)


In [ ]:
post = [neuron.skeleton_id for i, neuron in post_neurons_all_from_ann_df_filterby_today_df.iterrows()]

# Get all connections for the neurons
connections = pymaid.get_connectors(post, remote_instance=rm_octo)
display(connections)

connector_ids = connections['connector_id'].tolist()
connector_details = pymaid.get_connector_details(connector_ids)
display(connector_details)

# Step 2: Rename `postsynaptic_to` to `skeleton_id` for clarity
connector_details = connector_details.rename(columns={'postsynaptic_to': 'skeleton_id'})

# Step 3: Convert skeleton_id to numeric type. Note this cannot work when > 1 skeleton_id is in list
# Extract the first (and presumably only) element from each list in the 'postsynaptic_to' column
connector_details['skeleton_id'] = connector_details['skeleton_id'].apply(lambda x: x[0] if x else None)

connector_details['skeleton_id'] = connector_details['skeleton_id'].astype(int)

# Step 4: Merge with connectors DataFrame (if needed)
connectors_with_skid = pd.merge(connections, connector_details[['connector_id', 'skeleton_id']], 
                                on='connector_id', how='left')
display(connectors_with_skid)

print(f"dtype connectors_with_skid['skeleton_id'].dtype: {connectors_with_skid['skeleton_id'].dtype}")
print(f"post_neurons_all_from_ann_df_filterby_today_df['skeleton_id'].dtype: {post_neurons_all_from_ann_df_filterby_today_df['skeleton_id'].dtype}")

# Convert to numeric type
post_neurons_all_from_ann_df_filterby_today_df['skeleton_id'] = post_neurons_all_from_ann_df_filterby_today_df['skeleton_id'].astype(int)

# Step 5: Merge with post_neurons_df. Note connector = pre
final_post_conn_df_after_trans_filt = pd.merge(connectors_with_skid, post_neurons_all_from_ann_df_filterby_today_df, 
                  on='skeleton_id', how='left', suffixes=('_connector', '_post'))

display(final_post_conn_df_after_trans_filt)


final_post_conn_df_after_trans_filt = final_post_conn_df_after_trans_filt.rename(columns={
    'x_connector': 'connector_x',
    'y_connector': 'connector_y',
    'z_connector': 'connector_z',
    'x_post': 'post_x',
    'y_post': 'post_y',
    'z_post': 'post_z'
})
print("result")
display(final_post_conn_df_after_trans_filt)


In [ ]:
# Assuming you have a DataFrame 'final_post_conn_df_after_trans_filt' with 'skeleton_id' columns
skeleton_ids = final_post_conn_df_after_trans_filt['skeleton_id'].tolist()

# Get annotations for skeletons
skeleton_annotations = pymaid.get_annotations(skeleton_ids)
print(skeleton_annotations)


# Assuming your annotations are in a dictionary called 'annotations'
def get_relevant_annotations(anno_list):
    return [a for a in anno_list if a not in ['cube2: pushed fp synapses', 'pushed false positives synapses', 
                                              'cube1 : pushed fp synapses', 'cube3: pushed fp synapses']]

# Convert annotations dictionary to DataFrame
anno_data = []
for skeleton_id, annotations in skeleton_annotations.items():
    anno_data.append({
        'skeleton_id': skeleton_id,
        'annotations': annotations
    })

anno_df = pd.DataFrame(anno_data)
display(anno_df)

# Convert skeleton_id to integer
anno_df['skeleton_id'] = anno_df['skeleton_id'].astype(int)

# Check number of rows before merge
print("Rows in final_post_conn_df_after_trans_filt before merge:", len(final_post_conn_df_after_trans_filt))
print("Rows in anno_df:", len(anno_df))

# Check for duplicate skeleton_ids in both DataFrames
print("\nDuplicate skeleton_ids in final_post_conn_df_after_trans_filt:")
print(final_post_conn_df_after_trans_filt['skeleton_id'].value_counts())
print("\nDuplicate skeleton_ids in anno_df:")
print(anno_df['skeleton_id'].value_counts())


# Add annotations to the `final_post_conn_df_after_trans_filt` DataFrame
final_post_conn_df_after_trans_filt = final_post_conn_df_after_trans_filt.merge(anno_df, on='skeleton_id', how='inner')

display(final_post_conn_df_after_trans_filt)

In [ ]:
# Figure out which cube it is and put it in a new column
# Define a function to extract cube information
def get_cube_info(annotations):
    if not isinstance(annotations, list):
        return 'unknown'
    for anno in annotations:
        if 'cube' in anno.lower():
            if 'cube1' in anno.lower():
                return 'cube1'
            elif 'cube2' in anno.lower():
                return 'cube2'
            elif 'cube3' in anno.lower():
                return 'cube3'
    return 'unknown'

# Add cube column
final_post_conn_df_after_trans_filt['cube'] = final_post_conn_df_after_trans_filt['annotations'].apply(get_cube_info)
display(final_post_conn_df_after_trans_filt)

# Add other_annotations to new column
def get_other_annotations(annotations):
    if not isinstance(annotations, list):
        return []
    return [anno for anno in annotations 
            if 'cube' not in anno.lower() 
            and 'pushed false positives synapses' not in anno.lower()]

# Add other_annotations column
final_post_conn_df_after_trans_filt['other_annotations'] = final_post_conn_df_after_trans_filt['annotations'].apply(get_other_annotations)

display(final_post_conn_df_after_trans_filt)

# save to csv
# final_post_conn_df_after_trans_filt.to_csv(f"{output_dir}/final_post_conn_df_after_trans_filt.csv", index=False)


In [ ]:
# Now link it back to the transactions in today_df_filt

# Group by user and label and count the number of transactions
grouped_total_df = today_df_filt.groupby(['user', 'label']).size().reset_index(name='counts')
display(grouped_total_df)
display(today_df_filt)


#  Merge based on matching coordinates
final_post_conn_df_after_trans_filt_merge = final_post_conn_df_after_trans_filt.merge(
    today_df_filt[['post_x', 'post_y', 'post_z', 'user', 'user_id', 'project_id', 'transaction_id', 'execution_time', 'label']],
    left_on=['post_x', 'post_y', 'post_z'],
    right_on=['post_x', 'post_y', 'post_z'],
    how='inner'
)

display(final_post_conn_df_after_trans_filt_merge)

# group by user and label and count the number of transactions
grouped = final_post_conn_df_after_trans_filt_merge.groupby(['user']).size().reset_index(name='counts')
display(grouped)

# save csv file
final_post_conn_df_after_trans_filt_merge.to_csv(f"{output_dir}/final_df_postsyn_transaction_octo.csv", index=False)



In [ ]:
# group by user and label and count the number of transactions
grouped = final_post_conn_df_after_trans_filt_merge.groupby(['user', 'cube']).size().reset_index(name='counts')
display(grouped)

# find which users overlap in  'post_x', 'post_y', 'post_z'
# Find overlapping coordinates
grouped_overlap = final_post_conn_df_after_trans_filt_merge.groupby(['post_x', 'post_y', 'post_z']).size().reset_index(name='counts')
overlapping_coords = grouped_overlap[grouped_overlap['counts'] > 1]
display(overlapping_coords)


# Get the details for these overlapping coordinates
overlapping_details = final_post_conn_df_after_trans_filt_merge[
    final_post_conn_df_after_trans_filt_merge[['post_x', 'post_y', 'post_z']].apply(tuple, axis=1).isin(
        overlapping_coords[['post_x', 'post_y', 'post_z']].apply(tuple, axis=1)
    )
][['user', 'cube', 'post_x', 'post_y', 'post_z', 'other_annotations']]

# Display the results
display(overlapping_details.sort_values(['post_x', 'post_y', 'post_z']))

# MR1.4-3

In [ ]:
rm_mr = pymaid.CatmaidInstance(volumes[0]["url"], project_id=volumes[0]["project"], api_token=volumes[0]["token"],
                               http_user=http_user, http_password=http_pass)

rm_mr


In [ ]:
# Get transactions for MR143

project_id = volumes[1]["project"]

# Now call the function to retrieve enriched transactions.
try:
    txns_with_locations = get_transactions_with_locations(rm_mr, project_id, range_length=500)
    print(txns_with_locations)
    for txn in txns_with_locations:
        print("Transaction ID:", txn.get("transaction_id"))
        print("Location Data:", txn.get("location_data"))
        print("=" * 40)
except Exception as e:
    print("An error occurred:", e)

In [ ]:
import pandas as pd
output_dir = "/Users/sam/Library/CloudStorage/OneDrive-UniversityofCambridge/Synapse_localisation/synapse_curation/catmaid_tracker_plots"

# Convert to DataFrame
df = pd.DataFrame(txns_with_locations)

# Print the DataFrame
display(df)

# Save the DataFrame to a CSV file
df.to_csv(f"{output_dir}/transactions_with_locations_mr143.csv", index=False)

# group by user and label and count the number of transactions
grouped = df.groupby(['user', 'label']).size().reset_index(name='counts')
grouped

In [ ]:
# Filter transactions on the hackathon day
import pandas as pd
from datetime import datetime


# Convert execution_time to datetime if it's not already
df['execution_time'] = pd.to_datetime(df['execution_time'])

# Get today's date
today = pd.Timestamp(2025, 3, 6).date() #datetime.now().date()
print(today)

# Filter for today's transactions
today_df = df[df['execution_time'].dt.date >= today]

# drop one user- smohinta-su
today_df = today_df[today_df['user'] != 'smohinta-su']
today_df = today_df[today_df['user'] != 'smohinta']


# replace hack_guest with sharris
today_df['user'] = today_df['user'].replace({'hack_guest': 'sharris'})

# replace realnames with anonymised reviewer names
today_df['user_anon'] = today_df['user'].replace({'mclayton': 'Reviewer 1', 'mrobbins': 'Reviewer 2', 'gmo': 'Reviewer 3', 'adulac': 'Reviewer 4',
                                             'nceffa': 'Reviewer 5',
                                             'sharris': 'Reviewer 6',
                                             'swilson': 'Reviewer 7',
                                             'shiyan': 'Reviewer 8',
                                             'hack_guest': 'Reviewer 9'
                                             }) 

display(today_df)


# group by user and label and count the number of transactions
grouped = today_df.groupby(['user', 'label']).size().reset_index(name='counts')
grouped # this looks correct now based on our expectations from talking to the reviewers


In [ ]:
# Filter only annotations:add from transactions because we are interested in synapse annotations

today_df_filt = today_df[today_df['label'] == 'annotations.add']
display(today_df_filt)


# Again group by user and label and count the number of transactions
grouped_filt = today_df_filt.groupby(['user', 'label']).size().reset_index(name='counts')
grouped_filt


In [198]:
# Save the filtered DataFrame to a CSV file
today_df_filt.to_csv(f"{output_dir}/transactions_with_locations_filtered_mr143.csv", index=False)

In [ ]:
# Check how many location_data = {'x': None, 'y': None, 'z': None} are there
display(today_df_filt['location_data'].value_counts())

# Which users have location_data = {'x': None, 'y': None, 'z': None}
location_missing = today_df_filt[today_df_filt['location_data'] == {'x': None, 'y': None, 'z': None}]
# group by user and label and count the number of transactions
grouped_missing = location_missing.groupby(['user', 'label']).size().reset_index(name='counts')
display(grouped_missing)


# Filter out all missing locations
today_df_filt = today_df_filt[today_df_filt['location_data'] != {'x': None, 'y': None, 'z': None}]
display(today_df_filt)

# Extract x, y, z coordinates into separate columns
today_df_filt['post_x'] = today_df_filt['location_data'].apply(lambda x: x['x'])
today_df_filt['post_y'] = today_df_filt['location_data'].apply(lambda x: x['y'])
today_df_filt['post_z'] = today_df_filt['location_data'].apply(lambda x: x['z'])
display(today_df_filt)


# Group by user and label and count the number of transactions
grouped_filt = today_df_filt.groupby(['user', 'label']).size().reset_index(name='counts')
grouped_filt


#### Find all locations based on annotations list of MR143

In [ ]:
# Find all neurons_ids and locations based on annotations list
import pandas as pd

annotations_list_mr143 = ["annotation:cube1MBONj2: pushed fp synapses", "annotation:cube2likelyj2: pushed fp synapses"]

def get_neurons_from_annotations_list(annotations_list, remote_instance):
    """
    Get all neurons and their locations based on a list of annotations.
    """
    neurons = []
    for annotation in annotations_list:
        # Get all annotations with the given name
        annotation_neurons = pymaid.get_neurons(annotation, remote_instance=remote_instance)
        neurons.extend(annotation_neurons)
    return neurons

neurons = get_neurons_from_annotations_list(annotations_list_mr143, rm_mr)
print(neurons)

# Assuming these are all post neurons 
post = [neuron.id for neuron in neurons]
for i, neuron in enumerate(neurons):
    node_coords = neuron.nodes[['x', 'y', 'z']]
    print(f"Coordinates for neuron {neuron.skeleton_id}:")
    print(node_coords)
    print("=" * 40)
    if i == 2:
        break


# Create a list to store all neuron data
neuron_data = []

for neuron in neurons:
    # Get coordinates for each node in the neuron
    node_coords = neuron.nodes[['x', 'y', 'z']]
    
    # For each node in the neuron, create a row with all information
    for idx, coords in node_coords.iterrows():
        neuron_data.append({
            'neuron_id': neuron.id,
            'skeleton_id': neuron.skeleton_id,
            'name': neuron.name,
            'x': coords['x'],
            'y': coords['y'],
            'z': coords['z'],
            'node_id': idx
        })

# Convert to DataFrame
post_neurons_all_from_ann_df = pd.DataFrame(neuron_data)
display(post_neurons_all_from_ann_df)

In [ ]:
# Filter the post_neurons_all_from_ann_df based on the post neurons we have from today_df_filt based on locations
# Create a mask for matching coordinates
matching_coords = post_neurons_all_from_ann_df.apply(
    lambda row: any((row['x'] == today_df_filt['post_x']) & 
                   (row['y'] == today_df_filt['post_y']) & 
                   (row['z'] == today_df_filt['post_z'])), 
    axis=1
)

# Filter post_neurons_all_from_ann_df to keep only the matching rows
post_neurons_all_from_ann_df_filterby_today_df = post_neurons_all_from_ann_df[matching_coords]
display(post_neurons_all_from_ann_df_filterby_today_df)


In [ ]:
post = [neuron.skeleton_id for i, neuron in post_neurons_all_from_ann_df_filterby_today_df.iterrows()]

# Get all connections for the neurons
connections = pymaid.get_connectors(post, remote_instance=rm_mr)
display(connections)

connector_ids = connections['connector_id'].tolist()
print(connector_ids)
connector_details = pymaid.get_connector_details(connector_ids)
display(connector_details)

# Step 2: Rename `postsynaptic_to` to `skeleton_id` for clarity
connector_details = connector_details.rename(columns={'postsynaptic_to': 'skeleton_id'})

# Step 3: Convert skeleton_id to numeric type. Note this cannot work when > 1 skeleton_id is in list
# Extract the first (and presumably only) element from each list in the 'postsynaptic_to' column
connector_details['skeleton_id'] = connector_details['skeleton_id'].apply(lambda x: x[0] if x else None)

connector_details['skeleton_id'] = connector_details['skeleton_id'].astype(int)

# Step 4: Merge with connectors DataFrame (if needed)
connectors_with_skid = pd.merge(connections, connector_details[['connector_id', 'skeleton_id']], 
                                on='connector_id', how='left')
display(connectors_with_skid)

print(f"dtype connectors_with_skid['skeleton_id'].dtype: {connectors_with_skid['skeleton_id'].dtype}")
print(f"post_neurons_all_from_ann_df_filterby_today_df['skeleton_id'].dtype: {post_neurons_all_from_ann_df_filterby_today_df['skeleton_id'].dtype}")

# Convert to numeric type
post_neurons_all_from_ann_df_filterby_today_df['skeleton_id'] = post_neurons_all_from_ann_df_filterby_today_df['skeleton_id'].astype(int)

# Step 5: Merge with post_neurons_df. Note connector = pre
final_post_conn_df_after_trans_filt = pd.merge(connectors_with_skid, post_neurons_all_from_ann_df_filterby_today_df, 
                  on='skeleton_id', how='left', suffixes=('_connector', '_post'))

display(final_post_conn_df_after_trans_filt)


final_post_conn_df_after_trans_filt = final_post_conn_df_after_trans_filt.rename(columns={
    'x_connector': 'connector_x',
    'y_connector': 'connector_y',
    'z_connector': 'connector_z',
    'x_post': 'post_x',
    'y_post': 'post_y',
    'z_post': 'post_z'
})
print("result")
display(final_post_conn_df_after_trans_filt)


In [ ]:
# Assuming you have a DataFrame 'final_post_conn_df_after_trans_filt' with 'skeleton_id' columns
skeleton_ids = final_post_conn_df_after_trans_filt['skeleton_id'].tolist()

# Get annotations for skeletons
skeleton_annotations = pymaid.get_annotations(skeleton_ids, remote_instance=rm_mr)
print(skeleton_annotations)


# Assuming your annotations are in a dictionary called 'annotations'
def get_relevant_annotations(anno_list):
    return [a for a in anno_list if a not in ['cube2: pushed fp synapses', 'pushed false positives synapses', 
                                              'cube1 : pushed fp synapses', 'cube3: pushed fp synapses']]

# Convert annotations dictionary to DataFrame
anno_data = []
for skeleton_id, annotations in skeleton_annotations.items():
    anno_data.append({
        'skeleton_id': skeleton_id,
        'annotations': annotations
    })

anno_df = pd.DataFrame(anno_data)
display(anno_df)

# Convert skeleton_id to integer
anno_df['skeleton_id'] = anno_df['skeleton_id'].astype(int)

# Check number of rows before merge
print("Rows in final_post_conn_df_after_trans_filt before merge:", len(final_post_conn_df_after_trans_filt))
print("Rows in anno_df:", len(anno_df))

# Check for duplicate skeleton_ids in both DataFrames
print("\nDuplicate skeleton_ids in final_post_conn_df_after_trans_filt:")
print(final_post_conn_df_after_trans_filt['skeleton_id'].value_counts())
print("\nDuplicate skeleton_ids in anno_df:")
print(anno_df['skeleton_id'].value_counts())


# Add annotations to the `final_post_conn_df_after_trans_filt` DataFrame
final_post_conn_df_after_trans_filt = final_post_conn_df_after_trans_filt.merge(anno_df, on='skeleton_id', how='inner')

display(final_post_conn_df_after_trans_filt)

In [ ]:
# Figure out which cube it is and put it in a new column
# Define a function to extract cube information
def get_cube_info(annotations):
    if not isinstance(annotations, list):
        return 'unknown'
    for anno in annotations:
        if 'cube' in anno.lower():
            if 'cube1' in anno.lower():
                return 'cube1'
            elif 'cube2' in anno.lower():
                return 'cube2'
            elif 'cube3' in anno.lower():
                return 'cube3'
    return 'unknown'

# Add cube column
final_post_conn_df_after_trans_filt['cube'] = final_post_conn_df_after_trans_filt['annotations'].apply(get_cube_info)
display(final_post_conn_df_after_trans_filt)

# Add other_annotations to new column
def get_other_annotations(annotations):
    if not isinstance(annotations, list):
        return []
    return [anno for anno in annotations 
            if 'cube' not in anno.lower() 
            and 'pushed false positives synapses' not in anno.lower()]

# Add other_annotations column
final_post_conn_df_after_trans_filt['other_annotations'] = final_post_conn_df_after_trans_filt['annotations'].apply(get_other_annotations)

display(final_post_conn_df_after_trans_filt)

# save to csv
# final_post_conn_df_after_trans_filt.to_csv(f"{output_dir}/final_post_conn_df_after_trans_filt_mr143.csv", index=False)


In [ ]:
# Now link it back to the transactions in today_df_filt

# Group by user and label and count the number of transactions
grouped_total_df = today_df_filt.groupby(['user', 'label']).size().reset_index(name='counts')
display(grouped_total_df)
display(today_df_filt)


#  Merge based on matching coordinates
final_post_conn_df_after_trans_filt_merge = final_post_conn_df_after_trans_filt.merge(
    today_df_filt[['post_x', 'post_y', 'post_z', 'user', 'user_id', 'project_id', 'transaction_id', 'execution_time', 'label']],
    left_on=['post_x', 'post_y', 'post_z'],
    right_on=['post_x', 'post_y', 'post_z'],
    how='inner'
)

display(final_post_conn_df_after_trans_filt_merge)

# group by user and label and count the number of transactions
grouped = final_post_conn_df_after_trans_filt_merge.groupby(['user']).size().reset_index(name='counts')
display(grouped)

# save csv file
# final_post_conn_df_after_trans_filt_merge.to_csv(f"{output_dir}/final_df_postsyn_transaction_mr143.csv", index=False)



In [ ]:
# group by user and label and count the number of transactions
grouped = final_post_conn_df_after_trans_filt_merge.groupby(['user', 'cube']).size().reset_index(name='counts')
display(grouped)

# find which users overlap in  'post_x', 'post_y', 'post_z'
# Find overlapping coordinates
grouped_overlap = final_post_conn_df_after_trans_filt_merge.groupby(['post_x', 'post_y', 'post_z']).size().reset_index(name='counts')
overlapping_coords = grouped_overlap[grouped_overlap['counts'] > 1]
display(overlapping_coords)


# Get the details for these overlapping coordinates
overlapping_details = final_post_conn_df_after_trans_filt_merge[
    final_post_conn_df_after_trans_filt_merge[['post_x', 'post_y', 'post_z']].apply(tuple, axis=1).isin(
        overlapping_coords[['post_x', 'post_y', 'post_z']].apply(tuple, axis=1)
    )
][['user', 'cube', 'post_x', 'post_y', 'post_z', 'other_annotations']]

# Display the results
display(overlapping_details.sort_values(['post_x', 'post_y', 'post_z']))